Universidad de Buenos Aires

CEIA

Aprendizaje por Refuerzo II

Tema: **Aprendizaje por refuerzo multiagente (MAPPO) en fútbol 2 vs 2**


---

# Trabajo Práctico 1

De acuerdo a la siguiente celda:
1. Resolver y responder las preguntas modificando el código original y reescribiéndolo luego de cada pregunta.
2. **Ejecutar** cada celda modificada con `run_config` antes de responder cada pregunta (la función repite el entrenamiento con las semillas de la lista `SEEDS` y reporta la media y la desviación estándar). Cada entrenamiento es costoso: por defecto se usa **1 semilla**; si el tiempo lo permite, use `SEEDS = [0, 1]` o más para conclusiones más confiables.


**Requisito:** activar una GPU en `Entorno de ejecución > Cambiar tipo de entorno de ejecución`.

**Entorno:** `football` de VMAS. Dos equipos de 2 jugadores: el equipo **azul** (el que se entrena) juega contra el equipo **rojo**, controlado por una IA heurística de VMAS. Cada episodio dura como máximo 500 pasos y termina cuando se marca un gol. Las acciones son discretas (9 acciones de movimiento, sin disparo).

**Algoritmo:** MAPPO (Multi-Agent PPO) con CTDE.

**Métricas** (evaluación determinista contra la IA de dificultad máxima, `DIF_REF = 1.0`, siempre la misma para todas las configuraciones; 100 episodios al final del entrenamiento):
- *Goles a favor:* porcentaje de episodios en los que el equipo azul marca un gol.
- *Goles en contra:* porcentaje de episodios en los que la IA marca un gol.
- *Diferencia de gol:* goles a favor menos goles en contra (mayor es mejor). Los episodios restantes terminan sin gol.
- *Tiempo de entrenamiento* (incluye las evaluaciones periódicas).

**Parámetros que se pueden modificar** (diccionario `BASE_CFG`): `max_frames`, `lr`, `gamma`, `gae_lambda`, `clip_epsilon`, `entropia`, `n_envs`, `n_epochs`, `minibatch_size`, `dificultad_ia`, `recompensa_gol`, `shaping_balon`, `shaping_jugador`.


**Gráficos:** se solicitan únicamente en las preguntas 1, 2 y 7. En las demás preguntas analizar los resultados que imprime `run_config`.

**Dato:** las modificaciones son **acumulativas**. En cada pregunta se parte de la mejor configuración obtenida en las preguntas anteriores y se cambia únicamente lo que se solicita.

In [ ]:
# ============================================================
# 1: Instalación e imports
# ============================================================
# Si Colab solicita reiniciar el entorno de ejecución al terminar la instalación,
# reinícielo y vuelva a ejecutar esta celda.
!pip install -q vmas==1.5.2 benchmarl==1.5.2

import os
import re
import gc
import time
import numpy as np
import torch
import matplotlib.pyplot as plt
plt.style.use('dark_background')

import vmas
import benchmarl as _b
import torchrl as _t
from tensordict import TensorDict
from torchrl.envs.utils import set_exploration_type, ExplorationType
from benchmarl.algorithms import MappoConfig
from benchmarl.environments import VmasTask
from benchmarl.experiment import Experiment, ExperimentConfig
from benchmarl.experiment.callback import Callback
from benchmarl.models.mlp import MlpConfig

# ============================================================
# 2: Entorno y parámetros
# ============================================================
INTEGRANTES = "Nombre Apellido"   # se guarda dentro de cada archivo .pt
SEEDS = [0]                       # semillas por configuración (cada entrenamiento es costoso)
N_EVAL_EPISODES = 100             # episodios de la evaluación final
DIF_REF = 1.0                     # dificultad de la IA usada SIEMPRE para evaluar (1.0 = la más difícil)
FRAMES_POR_BATCH = 6000           # frames recolectados por iteración (fijo)
EQUIPOS_DIR = "equipos"           # carpeta donde se guardan los archivos .pt
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(EQUIPOS_DIR, exist_ok=True)

# ---- FIJO: debe ser idéntico para todos los equipos (y para el cuaderno de inferencia) ----
TASK_FIJO = dict(
    max_steps=500,
    n_blue_agents=2, n_red_agents=2,
    ai_red_agents=True, ai_blue_agents=False,
    enable_shooting=False,            # acciones: solo movimiento (9 acciones discretas)
    observe_teammates=True, observe_adversaries=True, dict_obs=False,
)
TASK_REF = dict(TASK_FIJO, ai_strength=DIF_REF)   # rival de evaluación

# Configuración base: funciona, pero NO está optimizada.
BASE_CFG = dict(
    max_frames=300_000,        # frames totales de entrenamiento (múltiplo de 6000)
    lr=1e-3,                   # tasa de aprendizaje
    gamma=0.9,                 # factor de descuento
    gae_lambda=0.9,            # lambda de GAE
    clip_epsilon=0.2,          # recorte de PPO
    entropia=0.05,             # coeficiente de entropía (más alto = explora más)
    n_envs=10,                 # entornos en paralelo (debe dividir a 6000: 10, 30, 50, 60, 100...)
    n_epochs=45,               # épocas de optimización por iteración
    minibatch_size=400,        # tamaño del minibatch
    dificultad_ia=1.0,         # velocidad de la IA rival durante el ENTRENAMIENTO (0..1)
    recompensa_gol=100.0,      # recompensa al marcar
    shaping_balon=10.0,        # premio por acercar el balón a la portería rival
    shaping_jugador=0.1,       # premio por acercarse al balón
    eval_cada_iters=10,        # cada cuántas iteraciones se mide el desempeño durante el entrenamiento
    eval_episodios=20,         # episodios por medición
    device=DEVICE,
)

_env = vmas.make_env(scenario="football", num_envs=1, device="cpu", continuous_actions=False, seed=1, **TASK_FIJO)
OBS_DIM = int(_env.reset()[0].shape[-1])
print("Observación por agente:", OBS_DIM, "| Acciones discretas: 9")
print("Dispositivo:", "GPU (" + torch.cuda.get_device_name(0) + ")" if torch.cuda.is_available() else "solo CPU (active una GPU)")

# ============================================================
# 3: Funciones de apoyo (evaluación contra la IA)
# ============================================================
def _slug(texto):
    return re.sub(r"[^\w\-]+", "_", texto).strip("_")

def elegir_acciones(policy, obs_list, grupo):
    obs = torch.stack(obs_list, dim=1)                    # [envs, agentes, obs_dim]
    n, na = obs.shape[0], obs.shape[1]
    td = TensorDict({grupo: TensorDict({"observation": obs}, batch_size=[n, na])}, batch_size=[n])
    with torch.no_grad(), set_exploration_type(ExplorationType.DETERMINISTIC):
        td = policy(td)
    return td.get((grupo, "action")).reshape(n, na)

def crear_env_eval(n_envs, task_kwargs, seed=0):
    return vmas.make_env(scenario="football", num_envs=n_envs, device=DEVICE,
                         continuous_actions=False, seed=seed, **task_kwargs)

def evaluar_vs_ia(policy, env, grupo):
    # Juega env.num_envs episodios en paralelo (1 por entorno) y cuenta goles
    n = env.num_envs; Lp = env.scenario.pitch_length
    obs = env.reset()
    terminado = torch.zeros(n, dtype=torch.bool, device=DEVICE)
    resultado = torch.zeros(n, dtype=torch.long, device=DEVICE)     # 1 gol a favor, -1 en contra, 0 sin gol
    retorno = torch.zeros(n, device=DEVICE)
    for _ in range(TASK_FIJO["max_steps"]):
        acc = elegir_acciones(policy, obs, grupo)
        obs, rews, done, _ = env.step([acc[:, i:i+1] for i in range(2)])
        retorno += rews[0].reshape(n) * (~terminado)
        x = env.scenario.ball.state.pos[:, 0]
        nuevo = done.reshape(n) & ~terminado
        resultado[nuevo & (x > Lp/2)] = 1
        resultado[nuevo & (x < -Lp/2)] = -1
        terminado |= done.reshape(n)
        if terminado.all(): break
    return dict(n=n, favor=int((resultado == 1).sum()), contra=int((resultado == -1).sum()),
                sin_gol=int((resultado == 0).sum()), retorno=float(retorno.mean()))

# ============================================================
# 4: Callback de seguimiento durante el entrenamiento
# ============================================================
class MonitorGoles(Callback):
    # Cada `cada_iters` iteraciones mide goles a favor y en contra contra la IA de referencia.
    def __init__(self, cada_iters, n_eval, task_kwargs, hist):
        super().__init__()
        self.cada, self.n_eval, self.task_kwargs, self.hist = cada_iters, n_eval, task_kwargs, hist
        self.env = None

    def on_batch_collected(self, batch):
        it = self.experiment.n_iters_performed
        if it % self.cada != 0: return
        if self.env is None: self.env = crear_env_eval(self.n_eval, self.task_kwargs, seed=999)
        grupo = list(self.experiment.group_map.keys())[0]
        r = evaluar_vs_ia(self.experiment.policy, self.env, grupo)
        r["iter"] = it
        r["frames"] = it * self.experiment.config.on_policy_collected_frames_per_batch
        self.hist.append(r)

# ============================================================
# 5: Entrenamiento de una configuración con una semilla
# ============================================================
def guardar_equipo(label, seed, experiment, grupo, hist):
    # Mismo formato que el cuaderno de entrenamiento: sirve para el cuaderno de inferencia.
    nombre = f"{_slug(label)}_s{seed}"
    archivo = os.path.join(EQUIPOS_DIR, f"equipo_{nombre}.pt")
    torch.save({
        "formato": "mappo-football-2v2-v1",
        "equipo": nombre,
        "integrantes": INTEGRANTES,
        "grupo": grupo,
        "obs_dim": OBS_DIM,
        "n_acciones": 9,
        "frames_entrenados": int(experiment.total_frames),
        "versiones": {"vmas": vmas.__version__, "torchrl": _t.__version__},
        "historial": hist,
        "policy_state_dict": {k: v.detach().cpu() for k, v in experiment.policy.state_dict().items() if torch.is_tensor(v)},
    }, archivo)
    return archivo

def run_experiment(cfg, seed, label):
    assert FRAMES_POR_BATCH % cfg["n_envs"] == 0, "n_envs debe dividir a 6000 (10, 30, 50, 60, 100...)"
    assert cfg["max_frames"] % FRAMES_POR_BATCH == 0, "max_frames debe ser múltiplo de 6000"

    task = VmasTask.FOOTBALL.get_from_yaml()
    task.config.update(dict(
        TASK_FIJO,
        ai_strength=cfg["dificultad_ia"],
        scoring_reward=cfg["recompensa_gol"],
        pos_shaping_factor_ball_goal=cfg["shaping_balon"],
        pos_shaping_factor_agent_ball=cfg["shaping_jugador"],
    ))

    algorithm_config = MappoConfig.get_from_yaml()    # MAPPO: actores descentralizados + crítico centralizado
    algorithm_config.entropy_coef = cfg["entropia"]
    algorithm_config.clip_epsilon = cfg["clip_epsilon"]
    algorithm_config.lmbda = cfg["gae_lambda"]
    model_config = MlpConfig.get_from_yaml()           # actor: MLP 2x256 (fijo)
    critic_model_config = MlpConfig.get_from_yaml()    # crítico: solo se usa al entrenar

    ecfg = ExperimentConfig.get_from_yaml()
    ecfg.sampling_device = ecfg.train_device = ecfg.buffer_device = cfg["device"]
    ecfg.share_policy_params = True                    # fijo
    ecfg.prefer_continuous_actions = False             # acciones discretas (fijo)
    ecfg.max_n_frames = cfg["max_frames"]
    ecfg.max_n_iters = None
    ecfg.on_policy_collected_frames_per_batch = FRAMES_POR_BATCH
    ecfg.on_policy_n_envs_per_worker = cfg["n_envs"]
    ecfg.on_policy_n_minibatch_iters = cfg["n_epochs"]
    ecfg.on_policy_minibatch_size = cfg["minibatch_size"]
    ecfg.lr = cfg["lr"]
    ecfg.gamma = cfg["gamma"]
    ecfg.evaluation = False                            # medimos nosotros con el monitor
    ecfg.render = False
    ecfg.loggers = []
    ecfg.create_json = False
    ecfg.checkpoint_at_end = False
    ecfg.save_folder = os.path.join(os.getcwd(), "entrenamiento_tmp")
    os.makedirs(ecfg.save_folder, exist_ok=True)

    hist = []
    monitor = MonitorGoles(cfg["eval_cada_iters"], cfg["eval_episodios"], TASK_REF, hist)
    experiment = Experiment(
        task=task, algorithm_config=algorithm_config, model_config=model_config,
        critic_model_config=critic_model_config, seed=seed, config=ecfg, callbacks=[monitor],
    )
    grupo = list(experiment.group_map.keys())[0]

    t0 = time.time()
    experiment.run()
    train_time = time.time() - t0

    env_f = crear_env_eval(N_EVAL_EPISODES, TASK_REF, seed=2024)
    r = evaluar_vs_ia(experiment.policy, env_f, grupo)
    archivo = guardar_equipo(label, seed, experiment, grupo, hist)

    res = dict(t=[h["frames"] for h in hist],
               f=[100 * h["favor"] / h["n"] for h in hist],
               c=[100 * h["contra"] / h["n"] for h in hist],
               r=[h["retorno"] for h in hist],
               time=train_time,
               favor=100 * r["favor"] / r["n"], contra=100 * r["contra"] / r["n"],
               retorno=r["retorno"], archivo=archivo, seed=seed)
    res["diff"] = res["favor"] - res["contra"]
    del experiment, env_f
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return res

# ============================================================
# 6: Ejecución con varias semillas, reporte y visualización
# ============================================================
RESULTS = {}   # resumen (media y desviación) de cada configuración
RUNS = {}      # resultados y curvas por semilla de cada configuración
CONFIGS = {}   # configuración usada en cada etiqueta

def run_config(label, cfg, seeds=SEEDS):
    print(f"=== {label} ===")
    runs = []
    for sd in seeds:
        res = run_experiment(cfg, sd, label)
        runs.append(res)
        print(f"  semilla {sd} | goles a favor: {res['favor']:5.1f}% | en contra: {res['contra']:5.1f}% | "
              f"diferencia: {res['diff']:6.1f} | tiempo: {res['time']/60:5.1f} min")

    fav = np.array([r["favor"] for r in runs]); con = np.array([r["contra"] for r in runs])
    dif = np.array([r["diff"] for r in runs]); tim = np.array([r["time"] for r in runs])
    RESULTS[label] = dict(favor_mean=fav.mean(), favor_std=fav.std(), contra_mean=con.mean(), contra_std=con.std(),
                          diff_mean=dif.mean(), diff_std=dif.std(), time_mean=tim.mean())
    RUNS[label] = runs
    CONFIGS[label] = dict(cfg)
    print(f"\n✅ Goles a favor: {fav.mean():.1f}% ± {fav.std():.1f} | En contra: {con.mean():.1f}% ± {con.std():.1f} | "
          f"Diferencia: {dif.mean():.1f} ± {dif.std():.1f} | Tiempo medio: {tim.mean()/60:.1f} min")
    print(f"   Equipo(s) guardado(s) en '{EQUIPOS_DIR}/'\n")
    return runs

def select_best(candidates, new_label):
    # Elige la mejor configuración (mayor diferencia de gol; desempate por goles a favor) y la guarda con una nueva etiqueta.
    best = max(candidates, key=lambda l: (RESULTS[l]["diff_mean"], RESULTS[l]["favor_mean"]))
    RESULTS[new_label] = dict(RESULTS[best])
    RUNS[new_label] = RUNS[best]
    CONFIGS[new_label] = dict(CONFIGS[best])
    print(f"Mejor configuración entre {candidates}: '{best}' -> guardada como '{new_label}'")
    return dict(CONFIGS[best])

def plot_curves(labels, key="f", band=False):
    # Superpone las curvas (media de las semillas) de varias configuraciones durante el entrenamiento.
    # key="f": % de episodios con gol a favor | key="c": % con gol en contra | key="r": retorno medio por episodio
    # band=True agrega la banda Mín-Máx entre semillas (se ve con 2 o más semillas).
    names = {"f": "Episodios con gol a favor (%)", "c": "Episodios con gol en contra (%)",
             "r": "Retorno medio por episodio"}
    plt.figure(figsize=(10, 5))
    for l in labels:
        runs = [r for r in RUNS[l] if len(r[key]) > 0]
        if len(runs) == 0:
            print(f"'{l}' no tiene datos para key='{key}'.")
            continue
        L = min(len(r[key]) for r in runs)
        t = np.array(runs[0]["t"][:L])
        data = np.array([r[key][:L] for r in runs])
        line, = plt.plot(t, data.mean(axis=0), linewidth=2, marker="o", markersize=3, label=l)
        if band:
            plt.fill_between(t, data.min(axis=0), data.max(axis=0), color=line.get_color(), alpha=0.2)
    plt.xlabel('Frames de entrenamiento'); plt.ylabel(names[key])
    plt.title(names[key] + ' | contra la IA de referencia (dificultad ' + str(DIF_REF) + ')')
    plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_summary(labels=None):
    # Barras con goles a favor, goles en contra y tiempo de entrenamiento.
    if labels is None:
        labels = list(RESULTS.keys())
    if len(labels) == 0:
        print("Aún no hay resultados guardados.")
        return
    x = np.arange(len(labels))
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].bar(x, [RESULTS[l]["favor_mean"] for l in labels],
                yerr=[RESULTS[l]["favor_std"] for l in labels], color='limegreen', capsize=4)
    axes[0].set_ylabel('% de episodios con gol a favor'); axes[0].set_ylim(0, 105)
    axes[0].set_title('Goles a favor (evaluación final)')
    axes[1].bar(x, [RESULTS[l]["contra_mean"] for l in labels],
                yerr=[RESULTS[l]["contra_std"] for l in labels], color='tomato', capsize=4)
    axes[1].set_ylabel('% de episodios con gol en contra'); axes[1].set_ylim(0, 105)
    axes[1].set_title('Goles en contra (evaluación final)')
    axes[2].bar(x, [RESULTS[l]["time_mean"] / 60 for l in labels], color='dodgerblue')
    axes[2].set_ylabel('Minutos'); axes[2].set_title('Tiempo medio de entrenamiento')
    for ax in axes:
        ax.set_xticks(x); ax.set_xticklabels(labels, rotation=30, ha='right')
        ax.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

def descargar_equipo(label, semilla=None):
    # Descarga el archivo .pt de una configuración guardada (para el cuaderno de inferencia).
    if label not in RUNS:
        print(f"No hay resultados guardados con la etiqueta '{label}'.")
        return
    runs = RUNS[label]
    run = runs[0] if semilla is None else next((r for r in runs if r["seed"] == semilla), None)
    if run is None:
        print(f"La etiqueta '{label}' no tiene la semilla {semilla}.")
        return
    print("Descargando:", run["archivo"])
    from google.colab import files
    files.download(run["archivo"])

## Configuración base

Entrene la configuración base. Sus resultados (etiqueta `Base`) son el punto de partida de todo el trabajo. Los números que imprime `run_config` son el punto de comparación para todas las preguntas.

In [ ]:
_ = run_config("Base", BASE_CFG)

**1.** La configuración base entrena al equipo contra una IA rival de dificultad máxima (`dificultad_ia=1.0`), pero el rival se mueve mucho más rápido que un equipo sin entrenar. ¿Cómo influye la dificultad de la IA rival durante el entrenamiento en lo que aprende el equipo? Pruebe al menos `dificultad_ia` en 0.6 y 0.3 y analice, con las métricas que imprime `run_config`, si conviene entrenar contra un rival más fácil. Recuerde que la evaluación siempre se hace contra la IA de dificultad 1.0.

**Gráficos solicitados:** curvas de goles a favor (`plot_curves(..., key="f")`) de `Base`, `P1a` y `P1b` en un mismo gráfico y gráfico de barras comparativo (`plot_summary`).

In [ ]:
# Pregunta 1: defina las configuraciones cfg_1a (dificultad_ia=0.6) y cfg_1b (dificultad_ia=0.3)
# a partir de BASE_CFG y ejecútelas con run_config("P1a", ...) y run_config("P1b", ...).
# Guarde la mejor configuración con select_best(["Base", "P1a", "P1b"], "P1") y genere los gráficos solicitados.

Respuesta

**2.** El `lr` base (1e-3) es mucho más alto que el habitual en PPO (BenchMARL usa 5e-5 por defecto) y se aplica durante 45 épocas de optimización en cada iteración. ¿Qué ocurre con el aprendizaje si se reduce el `lr` (por ejemplo a 3e-4, 1e-4 y 5e-5)? ¿Por qué? ¿Cómo cambia la estabilidad de las curvas?

**Gráficos solicitados:** curvas del retorno de entrenamiento de `P1` y de cada valor de `lr` probado en un mismo gráfico, con la banda Mín-Máx entre semillas (`plot_curves(..., key="r", band=True)`), y gráfico de barras comparativo (`plot_summary`).

In [ ]:
# Pregunta 2: parta de la mejor configuración de la pregunta 1 (cfg_1), pruebe distintos valores de lr
# (P2a, P2b, P2c). Guarde la mejor configuración con select_best(..., "P2") y genere los gráficos solicitados.

Respuesta

**3.** En cada iteración se recolectan 6000 frames, repartidos entre los `n_envs` entornos en paralelo (cada entorno aporta `6000 / n_envs` pasos). Con `n_envs=10` cada entorno recorre 600 pasos por iteración, pero con `n_envs=100` solo 60, y los episodios duran hasta 500 pasos. ¿Qué ocurre con el tiempo de entrenamiento y con la calidad del equipo si se aumenta `n_envs`? Pruebe al menos 30, 60 y 100 y justifique cuál es más conveniente con los resultados que imprime `run_config`.

In [ ]:
# Pregunta 3: parta de la mejor configuración de la pregunta 2 (cfg_2) y pruebe distintos valores de n_envs
# (deben dividir a 6000). Guarde la mejor configuración con select_best(..., "P3").

Respuesta

**4.** El coeficiente de entropía base (`entropia=0.05`) es alto para un problema con recompensa densa y un rival que debe ser superado con jugadas precisas. ¿Qué ocurre si se reduce a 0.01, 0.001 y 0.0? ¿Por qué? Relacione la respuesta con el hecho de que la evaluación se hace con la política determinista (la acción más probable).

In [ ]:
# Pregunta 4: parta de la mejor configuración de la pregunta 3 (cfg_3) y pruebe entropia = 0.01, 0.001 y 0.0
# (P4a, P4b, P4c). Guarde la mejor configuración con select_best(..., "P4").

Respuesta

**5.** Los episodios duran hasta 500 pasos y el gol (recompensa grande) suele ocurrir tras muchos pasos de juego. Con `gamma=0.9` el horizonte efectivo es de unos 10 pasos (aproximadamente `1 / (1 - gamma)`). ¿Qué ocurre si se aumenta `gamma` a 0.95, 0.99 y 0.995? ¿Por qué? Explique qué compromiso entre sesgo y varianza introduce un `gamma` más cercano a 1.

In [ ]:
# Pregunta 5: parta de la mejor configuración de la pregunta 4 (cfg_4) y pruebe gamma = 0.95, 0.99 y 0.995
# (P5a, P5b, P5c). Guarde la mejor configuración con select_best(..., "P5").

Respuesta

**6.** El equipo recibe una recompensa por gol (`recompensa_gol=100`) y dos premios densos de *shaping*: acercar el balón a la portería rival (`shaping_balon=10`) y acercarse al balón (`shaping_jugador=0.1`). ¿Qué ocurre si se modifican los pesos del *shaping*? Pruebe al menos tres combinaciones (por ejemplo: `shaping_balon` en 0, 3 y 30, y/o `shaping_jugador` en 0 y 1) y explique si el *shaping* ayuda a aprender o puede inducir comportamientos no deseados (por ejemplo, mover el balón sin llegar a marcar o descuidar la defensa).

In [ ]:
# Pregunta 6: parta de la mejor configuración de la pregunta 5 (cfg_5) y pruebe al menos tres combinaciones de
# shaping_balon y shaping_jugador (P6a, P6b, P6c, ...). Guarde la mejor configuración con select_best(..., "P6").

Respuesta

**7.** Con la mejor configuración de la pregunta 6, ¿qué cantidad de `max_frames` conviene usar si se prueba con:

- 300000 (ya obtenido en `P6`)
- 600000
- 1200000

¿Mejora, empeora o se mantiene aproximadamente igual el desempeño al aumentar los frames? ¿Desde qué cantidad de frames deja de compensar el tiempo adicional? ¿Qué tasa de goles a favor y en contra alcanza la configuración final (`P7`)? Para los valores más grandes se permite usar una sola semilla (`seeds=[0]`) por razones de tiempo; indíquelo en la respuesta.

**Gráficos solicitados:**
- (i) Curvas de goles a favor para cada cantidad de frames (`plot_curves(..., key="f")`).
- (ii) Diferencia de gol final y tiempo de entrenamiento en función de `max_frames` (use los valores guardados en `RESULTS`).
- (iii) Gráfico comparativo de todas las etapas del trabajo, `Base`, `P1`, ..., `P7` (`plot_summary`). Para ello guarde la mejor configuración de cada pregunta con `select_best(..., "P<n>")`.

**Partido entre dos equipos (inferencia):**
1. El **Equipo A** es la configuración final `P7`. Cree el **Equipo B** como una configuración distinta de `P7` en **al menos tres hiperparámetros** (por ejemplo `lr`, `entropia`, `gamma`, `clip_epsilon`, `n_epochs`, `dificultad_ia` o los pesos de *shaping*) y con los mismos `max_frames`. Entrénelo con `run_config("Equipo_B", cfg_B)`.
2. Descargue ambos archivos con `descargar_equipo("P7")` y `descargar_equipo("Equipo_B")`.
3. Abra el cuaderno de inferencia `rl2_tp1_2_inferencia_futbol_2v2_mappo_v3.ipynb`, suba el archivo del Equipo A como equipo AZUL y el del Equipo B como equipo ROJO, y ejecute el partido (sección 3) y la serie de partidos con intercambio de lados (sección 4).
4. **Adjunte al TP la animación del partido** (`partido.mp4`, que se guarda en los archivos de la sesión de Colab) y copie en la respuesta el marcador y el resultado de la serie.

¿Qué equipo gana el partido y la serie? ¿Coincide el resultado con lo que predecían las métricas contra la IA? Saque conclusiones relacionando el resultado con las diferencias de hiperparámetros entre ambos equipos. Si el cuaderno de inferencia falla o produce resultados incoherentes, corrija el error y descríbalo.

In [ ]:
# Pregunta 7: parta de la mejor configuración de la pregunta 6 (cfg_6) y pruebe distintos valores de max_frames.
# Guarde la mejor configuración con select_best(..., "P7") y genere los gráficos solicitados.
#
# Luego defina cfg_B (distinta de P7 en al menos tres hiperparámetros), entrénela con
# run_config("Equipo_B", cfg_B), descargue ambos equipos con descargar_equipo("P7") y
# descargar_equipo("Equipo_B") y ejecute el cuaderno de inferencia para enfrentarlos.

Respuesta